In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install ultralytics -q
import torch, os

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Verify inputs
print("\n=== /kaggle/input contents ===")
for item in sorted(os.listdir('/kaggle/input')):
    print(f"  {item}")
    subpath = f'/kaggle/input/{item}'
    if os.path.isdir(subpath):
        for sub in sorted(os.listdir(subpath))[:5]:
            print(f"    {sub}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.0 MB/s eta 0:00:00
GPU: Tesla T4
Torch: 2.10.0+cu128
CUDA available: True

=== /kaggle/input contents ===
  datasets
    moazelkhashab
    mohieymohamed


In [2]:
import os

print("=== All inputs ===")
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    if depth <= 2:
        indent = '  ' * depth
        print(f"{indent}{os.path.basename(root)}/")
        # نطبع الملفات بس لو فيها .pt أو .yaml
        for f in files:
            if f.endswith(('.pt', '.yaml', '.yml')):
                size_mb = os.path.getsize(os.path.join(root, f)) / 1024 / 1024
                print(f"{indent}  📄 {f} ({size_mb:.1f} MB)")

=== All inputs ===
input/
  datasets/
    moazelkhashab/
    mohieymohamed/


In [3]:
import os

print("=== FULL TREE SCAN ===\n")

# نمشي بعمق أكبر ونطبع كل الملفات المهمة
for root, dirs, files in os.walk('/kaggle/input'):
    # print الـ directory
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root) or 'input'}/")
    
    # print الملفات المهمة
    sub_indent = '  ' * (level + 1)
    for f in files:
        full_path = os.path.join(root, f)
        try:
            size = os.path.getsize(full_path)
            if size > 1024 * 1024:
                size_str = f"{size / 1024 / 1024:.1f} MB"
            else:
                size_str = f"{size / 1024:.1f} KB"
            print(f"{sub_indent}📄 {f} ({size_str})")
        except:
            print(f"{sub_indent}📄 {f}")

print("\n\n=== LOOKING FOR KEY FILES ===\n")

# نبحث عن الـ files المحددة
import glob

for pattern, label in [
    ('/kaggle/input/**/last.pt', 'Checkpoint (last.pt)'),
    ('/kaggle/input/**/best.pt', 'Checkpoint (best.pt)'),
    ('/kaggle/input/**/data_cleaned.yaml', 'Dataset YAML'),
    ('/kaggle/input/**/Heroglyphics_Signs', 'Dataset folder'),
]:
    matches = glob.glob(pattern, recursive=True)
    print(f"{label}:")
    if matches:
        for m in matches:
            print(f"   ✅ {m}")
    else:
        print(f"   ❌ NOT FOUND")
    print()

=== FULL TREE SCAN ===

input/
  datasets/
    moazelkhashab/
      tourasna-detector-v2-ckpt/
        📄 last.pt (150.9 MB)
        📄 best.pt (150.9 MB)
    mohieymohamed/
      heroglyphics-signs/
        Heroglyphics_Signs/
          📄 README.dataset.txt (0.2 KB)
          📄 README.roboflow.txt (0.9 KB)
          📄 data_cleaned.yaml (5.4 KB)
          valid/
            labels/
              📄 wall_section5301_1_jpg.rf.8fad11a5eb66cc03d58c4cd2f877399c.txt (4.7 KB)
              📄 VXiEq8W_2_png.rf.6b5b130fcc9ecac2f419c71250e4564a.txt (1.2 KB)
              📄 wall_section6540_10_png.rf.4707db2d0060f9507e5303db0904927a.txt (0.3 KB)
              📄 wall_section_538_png.rf.6b5fea12d796cff109569212d37e01cd.txt (0.7 KB)
              📄 wall_section_549_jpg.rf.29a4f7a8a900e02e5b159bbfbc259142.txt (7.2 KB)
              📄 wall_section10767_1_png.rf.e891daa56e1c6692fc0569b4cab2af3b.txt (0.4 KB)
              📄 wall_section5539_0_png.rf.f74fc29a36d909474009f1735af6f2d0.txt (0.2 KB)
            

In [4]:
import shutil, os

SRC = '/kaggle/input/datasets/mohieymohamed/heroglyphics-signs/Heroglyphics_Signs'
BASE = '/kaggle/working/dataset'

if not os.path.exists(BASE):
    print("📦 Copying dataset to working directory...")
    shutil.copytree(SRC, BASE)
    print("✅ Dataset copied")
else:
    print("✅ Dataset already exists")

# Clean: remove only lines with class >= 767
NC = 767
fixed_files = 0
removed_lines = 0

for split in ['train', 'valid', 'test']:
    lbl_dir = f'{BASE}/{split}/labels'
    for fname in os.listdir(lbl_dir):
        if not fname.endswith('.txt'):
            continue
        fpath = os.path.join(lbl_dir, fname)
        with open(fpath, 'r') as f:
            lines = f.readlines()
        clean = []
        bad = 0
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                bad += 1
                continue
            try:
                cls = int(parts[0])
            except ValueError:
                bad += 1
                continue
            if cls < NC:
                clean.append(line)
            else:
                bad += 1
        if bad > 0:
            with open(fpath, 'w') as f:
                f.writelines(clean)
            fixed_files += 1
            removed_lines += bad

print(f"\n✅ Cleaning done!")
print(f"   Fixed files:   {fixed_files}")
print(f"   Removed lines: {removed_lines}")

# Verify
print(f"\n--- Verification ---")
for split in ['train', 'valid', 'test']:
    imgs = len(os.listdir(f'{BASE}/{split}/images'))
    lbls = len(os.listdir(f'{BASE}/{split}/labels'))
    print(f"   {split:6s}: {imgs} images, {lbls} labels")

📦 Copying dataset to working directory...
✅ Dataset copied

✅ Cleaning done!
   Fixed files:   3381
   Removed lines: 48549

--- Verification ---
   train : 27937 images, 27937 labels
   valid : 3460 images, 3460 labels
   test  : 965 images, 965 labels


In [5]:
import yaml

BASE = '/kaggle/working/dataset'

with open(f'{BASE}/data_cleaned.yaml', 'r') as f:
    data_yaml = yaml.safe_load(f)

data_yaml['train'] = f'{BASE}/train/images'
data_yaml['val']   = f'{BASE}/valid/images'
data_yaml['test']  = f'{BASE}/test/images'

# ALSO need to handle `path` key (sometimes YOLO uses it)
data_yaml['path']  = BASE

with open(f'{BASE}/data_fixed.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print(f"✅ YAML ready")
print(f"   Classes: {data_yaml['nc']}")
print(f"   Names:   {len(data_yaml['names'])} classes")
print(f"   Train:   {data_yaml['train']}")
print(f"   Val:     {data_yaml['val']}")
print(f"   First 5 names: {data_yaml['names'][:5]}")
print(f"   Last 5 names:  {data_yaml['names'][-5:]}")

✅ YAML ready
   Classes: 767
   Names:   767 classes
   Train:   /kaggle/working/dataset/train/images
   Val:     /kaggle/working/dataset/valid/images
   First 5 names: ['A1', 'A10', 'A11', 'A12', 'A13']
   Last 5 names:  ['Z5', 'Z6', 'Z7', 'Z8', 'Z9']


In [ ]:
import shutil
import os
from ultralytics import YOLO

# Paths
CKPT_SRC = '/kaggle/input/datasets/moazelkhashab/tourasna-detector-v2-ckpt/last.pt'
CKPT_DST = '/kaggle/working/last.pt'

# Copy checkpoint to writable location (required for resume)
if not os.path.exists(CKPT_DST):
    print(f"📦 Copying checkpoint...")
    shutil.copy(CKPT_SRC, CKPT_DST)
    print(f"✅ Checkpoint copied: {os.path.getsize(CKPT_DST) / 1024 / 1024:.1f} MB")
else:
    print(f"✅ Checkpoint exists at {CKPT_DST}")+
    

# Load the model
print(f"\n🔄 Loading model from checkpoint...")
model = YOLO(CKPT_DST)

# Resume training
print(f"\n🚀 Starting resume training...")
print(f"   Will continue from epoch 18 → 50 (32 epochs remaining)")
print(f"   Estimated time: ~6-8 hours on T4\n")

results = model.train(
    resume=True,
    
    patience=20,        
    save_period=1,      
)

print("\n🎉 Training finished!")
print(f"Results saved to: {results.save_dir}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📦 Copying checkpoint...
✅ Checkpoint copied: 150.9 MB

🔄 Loading model from checkpoint...

🚀 Starting resume training...
   Will continue from epoch 18 → 50 (32 epochs remaining)
   Estimated time: ~6-8 hours on T4

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn

In [ ]:
import shutil

# Find the latest run
runs_dir = '/kaggle/working/runs'
if os.path.exists(runs_dir):
    # Get the most recent run
    all_runs = sorted(os.listdir(runs_dir), key=lambda x: os.path.getmtime(f'{runs_dir}/{x}'))
    latest = all_runs[-1]
    best_path = f'{runs_dir}/{latest}/weights/best.pt'
    last_path = f'{runs_dir}/{latest}/weights/last.pt'
    
    if os.path.exists(best_path):
        # Copy to /kaggle/working root for easy download
        shutil.copy(best_path, '/kaggle/working/best_resumed.pt')
        print(f"✅ Saved: /kaggle/working/best_resumed.pt")
    
    if os.path.exists(last_path):
        shutil.copy(last_path, '/kaggle/working/last_resumed.pt')
        print(f"✅ Saved: /kaggle/working/last_resumed.pt")
    
    # Copy results CSV too
    results_csv = f'{runs_dir}/{latest}/results.csv'
    if os.path.exists(results_csv):
        shutil.copy(results_csv, '/kaggle/working/training_results.csv')
        print(f"✅ Saved: training_results.csv")